# random forest vs xgboost

predicting house price with a random forest first, xgboost comes next. cleaned data comes from `eda.ipynb`.

In [1]:
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
SEED = 42

## 1. load data and hold out one listing

one random listing is taken out before any split. it is not used for training, tuning or testing, and i use it at the end to try the saved model on a house it has never seen.

In [2]:
df = pd.read_csv('../data/processed/houses_clean.csv')
df['is_member'] = df['is_member'].map({True: 1.0, False: 0.0, 'True': 1.0, 'False': 0.0})
print(df.shape)

holdout = df.sample(1, random_state=SEED)
df = df.drop(holdout.index).reset_index(drop=True)
print('rows left for training and testing:', len(df))

features = ['bedrooms', 'bathrooms', 'house_size_sqft', 'land_size_perches', 'posted_month',
            'is_verified', 'location', 'district', 'membership_level', 'is_member']
X = df[features]
y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(X_train.shape, X_test.shape)

(34254, 11)
rows left for training and testing: 34253
(27402, 10) (6851, 10)


## 2. preprocessing

numbers get the median for missing values and then scaling. categories (location, district, membership) get the most common value and one hot encoding. all of it sits in one `ColumnTransformer`, so it is fitted on the training rows only. price is skewed, so the model learns `log(price)` and the result is turned back into rupees.

In [3]:
num_cols = ['bedrooms', 'bathrooms', 'house_size_sqft', 'land_size_perches', 'posted_month', 'is_verified', 'is_member']
cat_cols = ['location', 'district', 'membership_level']

preprocess = ColumnTransformer([
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ]), num_cols),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), cat_cols),
])

def make_pipeline(regressor):
    model = TransformedTargetRegressor(regressor=regressor, func=np.log1p, inverse_func=np.expm1)
    return Pipeline([('prep', preprocess), ('model', model)])

## 3. a first random forest

a default random forest to see where i stand. it is scored with rmsle (rmse on log price), so a 10% miss counts the same for a cheap and an expensive house.

In [4]:
rf = make_pipeline(RandomForestRegressor(random_state=SEED, n_jobs=-1)).fit(X_train, y_train)
pred = rf.predict(X_test)
log_t, log_p = np.log1p(y_test), np.log1p(pred)
print('rmsle:', round(mean_squared_error(log_t, log_p) ** 0.5, 3))
print('r2 (log):', round(r2_score(log_t, log_p), 3))

rmsle: 0.349
r2 (log): 0.814


rmsle is 0.349 and r2 on log price is 0.814. that looks good, maybe too good. reposts of the same house are still in the data, so a house can be in train while its repost is in test. i should go back and clean that.